Simulation examples beyond validation.

- Squeeze-Out
- Customized Constitutive Laws

# Squeeze-Out

A parabolic plate approaches a soft elastic surface ($E = 1$ MPa) at constant velocity $V$. The gap height of the rigid geometry is

$$h(x,t) = H_0 - V t + A \left(x - x_c\right)^2, \qquad x_c = L_x/2,$$

with $H_0 = 30\,\mu \text{m}$, $V = 5$ mm/s and curvature $A = 0.25\,\mathrm{m^{-1}}$. Ambient pressure is prescribed at both ends.

The time-dependent geometry cannot be specified in the YAML file. It is set from the Python script below by a callback that updates the gap height after every time step.

$\rightarrow$ **[YAML File](../../tests/configs/parabolic_squeeze_elastic.yaml)**

In [ ]:
from GaPFlow.problem import Problem

H0, V, A = 30.0e-6, 5.0e-3, 25.0e-2

problem = Problem.from_yaml('../../tests/configs/parabolic_squeeze_elastic.yaml')
x_center = problem.grid['Lx'] / 2


def set_gap():
    problem.topo.set_mapped_height(H0 - V * problem.simtime + A * (problem.topo.xx - x_center) ** 2)

set_gap()  # this is annoying, will be fixed
problem.add_callback(set_gap)
problem.run()

In [ ]:
problem.animate()

# Customized Equation of State

A user-defined equation of state is registered with <code style="color:#c586c0">problem.set_custom_eos(function)</code>. The function takes the density $\rho$ (and optionally the gap height $h$) as scalars and returns the pressure in SI units.

Inclined slider with a compressible gas. Three cases are compared:

- built-in law
- custom law (same as built-in for validation)
- custom law with gap-height dependency - illustrative example without physical background

$\rightarrow$ **[YAML File](../../tests/configs/inclined_slider.yaml)**

In [ ]:
from utils.plotting import plot_custom_eos_comparison
from utils.solver_comparison import get_profile_1d

P0, rho0, h_min = 101325., 1.1853, 1.0e-5


def ideal_gas(rho):
    return P0 * rho / rho0


def gap_dependent(rho, h):
    return P0 * (rho / rho0) ** (1.0 + h_min / h)


results = {}
for label, eos in [('built-in', None), ('custom, same law', ideal_gas), ('custom, gap-dependent', gap_dependent)]:
    problem = Problem.from_yaml('../../tests/configs/inclined_slider.yaml')
    if eos is not None:
        problem.set_custom_eos(eos)  # the magic happens here
    problem.run()
    results[label] = get_profile_1d(problem)

plot_custom_eos_comparison(results)